# 이미지 속 글자를 문자열로 — OCR 파인튜닝 시연 (19_ocr_finetune_demo)

**교수용 시연 노트북**입니다. 학생은 결과 화면만 봅니다.

사진 속 글자를 읽어 **문자열로 바꾸는 일**을 OCR(Optical Character Recognition, 광학 문자 인식)이라고 합니다.
사전학습 OCR 모델 **TrOCR-small** 을 가져와, **부품 일련번호 라벨**(예: `KX-4047-3B`)을 읽도록 파인튜닝합니다.

```
  셀 0  설정
  셀 1  사전학습 모델 불러오기 (TrOCR-small, 약 6,200만 파라미터)
  셀 2  합성 데이터 만들기 — 글자를 그림으로 그려서 (이미지, 정답 문자열) 쌍을 만든다
  셀 3  파인튜닝 전 성능 — 사전학습 모델이 그대로 읽으면?
  셀 4  파인튜닝
  셀 5  파인튜닝 후 성능 — 전후 비교
  셀 6  새 이미지 읽어 보기
  셀 7  (선택) 모델 저장
```

| 구분 | 내용 |
|---|---|
| 입력 | 글자가 들어 있는 이미지 (가로로 긴 한 줄) |
| 출력 | 문자열 (`"KX-4047-3B"`) |
| 모델 | 이미지 인코더(ViT 계열) + 텍스트 디코더 — 번역 모델처럼 **글자 토큰을 하나씩 생성** |
| 데이터 | 다운로드 없음. 파이썬으로 직접 그려서 만든다 |

In [ ]:
# 셀 0 — 설정
import torch
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "microsoft/trocr-small-printed"   # 인쇄체 글자로 사전학습된 작은 OCR 모델
SEED       = 42
LR         = 5e-5                               # 사전학습 모델은 작은 학습률로 조금씩 고친다
if DEVICE == "cuda":
    N_TRAIN, N_VAL, EPOCHS, BATCH = 2000, 100, 3, 16   # 8GB GPU 기준 (메모리 부족 오류가 나면 BATCH = 8)
else:
    N_TRAIN, N_VAL, EPOCHS, BATCH = 320, 40, 1, 8      # CPU 는 느리므로 아주 작게
print(f"장치 {DEVICE} · 훈련 {N_TRAIN}장 · 검증 {N_VAL}장 · 에폭 {EPOCHS} · 배치 {BATCH}")

In [ ]:
# 셀 1 — 사전학습 모델 불러오기
import random, string, time
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# processor = 이미지 전처리기(384×384 로 맞추기) + 토크나이저(문자열 ↔ 토큰 번호)
# use_fast=False : protobuf 패키지 없이 sentencepiece 토크나이저를 쓴다
processor = TrOCRProcessor.from_pretrained(MODEL_NAME, use_fast=False)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(DEVICE)
tok = processor.tokenizer
START_ID, PAD_ID = model.generation_config.decoder_start_token_id, tok.pad_token_id   # 2, 1

print(f"파라미터 {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("인코더(이미지):", type(model.encoder).__name__, "· 디코더(글자):", type(model.decoder).__name__)
print("토큰으로 쪼개 보기:", tok.tokenize("KX-4047-3B"))

---

## 셀 2 — 합성 데이터

진짜 라벨 사진을 수천 장 찍기는 어렵습니다. 그래서 **정답 문자열을 먼저 정하고, 그 글자를 그림으로 그립니다.**
정답을 알고 그렸으니 라벨링이 필요 없습니다.
실제 사진처럼 보이도록 글꼴 · 배경색 · 기울기 · 흐림 · 잡선을 무작위로 섞습니다.

형식: `영문 2자 - 숫자 4자 - 영문·숫자 2자` → `O`(오)와 `0`(영), `Z`와 `2`, `B`와 `8` 처럼 **헷갈리는 글자**가 섞입니다.

In [ ]:
# 셀 2 — 합성 데이터 만들기
import os
FONT_DIR = "C:/Windows/Fonts"
FONTS = [os.path.join(FONT_DIR, f) for f in ["consola.ttf", "arial.ttf", "times.ttf", "cour.ttf", "arialbd.ttf"]]
FONTS = [f for f in FONTS if os.path.exists(f)]
print("사용할 글꼴:", [os.path.basename(f) for f in FONTS] or "기본 글꼴")

def make_text(r):
    a = "".join(r.choices(string.ascii_uppercase, k=2))
    b = "".join(r.choices(string.digits, k=4))
    c = "".join(r.choices(string.ascii_uppercase + string.digits, k=2))
    return f"{a}-{b}-{c}"

def draw_label(text, r):
    size = r.randint(26, 36)
    font = ImageFont.truetype(r.choice(FONTS), size) if FONTS else ImageFont.load_default(size=size)
    bg = tuple(r.randint(120, 230) for _ in range(3))                # 배경색
    fg = tuple(max(0, v - r.randint(60, 110)) for v in bg)          # 글자색 (배경보다 조금 어둡게)
    img = Image.new("RGB", (260, 56), bg)
    d = ImageDraw.Draw(img)
    d.text((r.randint(4, 20), r.randint(4, 12)), text, font=font, fill=fg)
    for _ in range(r.randint(3, 8)):                                 # 긁힌 자국 같은 잡선
        d.line([(r.randint(0, 260), r.randint(0, 56)), (r.randint(0, 260), r.randint(0, 56))], fill=fg, width=1)
    img = img.rotate(r.uniform(-4, 4), fillcolor=bg)                 # 살짝 기울기
    return img.filter(ImageFilter.GaussianBlur(r.uniform(0.3, 1.2)))  # 살짝 흐리게

def make_set(n, seed):
    r = random.Random(seed)
    return [(t := make_text(r), draw_label(t, r)) for _ in range(n)]

train_set = make_set(N_TRAIN, seed=1)
val_set   = make_set(N_VAL,   seed=0)      # 다른 시드 → 훈련에 없던 문자열
print(f"훈련 {len(train_set)}장 · 검증 {len(val_set)}장 · 예: {train_set[0][0]}")

fig, axes = plt.subplots(2, 4, figsize=(13, 3))
for ax, (t, im) in zip(axes.flat, train_set):
    ax.imshow(im); ax.set_title(t, fontsize=12); ax.axis("off")
plt.suptitle("합성 라벨 이미지와 정답 문자열"); plt.tight_layout(); plt.show()

---

## 셀 3 — 파인튜닝 전 성능

지표 두 개를 봅니다.

| 지표 | 뜻 |
|---|---|
| **문자열 정확도** | 10글자가 **전부** 맞아야 정답 (일련번호는 한 글자만 틀려도 쓸모없음) |
| **CER** (Character Error Rate, 글자 오류율) | 정답으로 고치려면 글자를 몇 개 바꾸고·넣고·빼야 하나 ÷ 정답 길이. **낮을수록 좋다** |

In [ ]:
# 셀 3 — 파인튜닝 전 성능
def to_pixels(images):
    return processor(images=images, return_tensors="pt").pixel_values.to(DEVICE)

@torch.no_grad()
def read_images(images, bs=32):
    """이미지 목록 → 문자열 목록"""
    model.eval(); out = []
    for i in range(0, len(images), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            ids = model.generate(to_pixels(images[i:i+bs]), max_new_tokens=20)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return [s.strip() for s in out]

def edit_distance(a, b):
    """a 를 b 로 바꾸는 데 필요한 최소 글자 수정 횟수"""
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def evaluate(data):
    preds = read_images([im for _, im in data])
    acc = sum(p == t for (t, _), p in zip(data, preds)) / len(data)
    cer = sum(edit_distance(p, t) for (t, _), p in zip(data, preds)) / sum(len(t) for t, _ in data)
    return acc, cer, preds

def show_examples(data, preds, title, n=8, only_wrong=False):
    pairs = [(t, im, p) for (t, im), p in zip(data, preds) if not (only_wrong and p == t)][:n]
    if not pairs: print("보여 줄 예가 없습니다"); return
    fig, axes = plt.subplots(2, 4, figsize=(13, 3.4))
    for ax in axes.flat: ax.axis("off")
    for ax, (t, im, p) in zip(axes.flat, pairs):
        ax.imshow(im)
        ax.set_title(f"정답 {t}\n예측 {p}", fontsize=10, color="green" if p == t else "red")
    plt.suptitle(title); plt.tight_layout(); plt.show()

t0 = time.time()
acc_before, cer_before, preds_before = evaluate(val_set)
print(f"[파인튜닝 전] 문자열 정확도 {acc_before*100:.0f}% · CER {cer_before*100:.1f}%  ({time.time()-t0:.0f}초)")
show_examples(val_set, preds_before, "파인튜닝 전 — 틀린 예 (빨강)", only_wrong=True)

---

## 셀 4 — 파인튜닝

디코더는 **앞 글자들을 보고 다음 글자를 맞히는** 방식으로 학습합니다 (문장 생성 모델과 같은 방식).

```
디코더 입력 :  </s>  KX   -40   47  ...   3B
맞힐 정답   :   KX   -40   47   ...  3B   </s>      ← 한 칸씩 밀려 있다
```

손실은 각 위치에서 "정답 토큰을 얼마나 높은 확률로 골랐나"의 교차 엔트로피 평균입니다.
(라이브러리 내장 손실은 이 버전에서 정답을 한 칸 더 밀어 버리므로, 여기서는 직접 계산합니다.)

In [ ]:
# 셀 4 — 파인튜닝
def make_batch(items):
    pixels = to_pixels([im for _, im in items])
    ids = tok([t for t, _ in items], padding=True, return_tensors="pt").input_ids[:, 1:]   # 맨 앞 <s> 는 뺀다
    labels = ids.masked_fill(ids == PAD_ID, -100).to(DEVICE)                               # 빈칸(패딩)은 채점 제외
    dec_in = torch.cat([torch.full((ids.size(0), 1), START_ID), ids[:, :-1]], dim=1).to(DEVICE)  # 한 칸 밀기
    return pixels, dec_in, labels

torch.manual_seed(SEED)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))   # fp16 으로 메모리·시간 절약
losses, t0 = [], time.time()
for ep in range(EPOCHS):
    model.train()
    random.Random(ep).shuffle(train_set)
    for i in range(0, len(train_set), BATCH):
        pixels, dec_in, labels = make_batch(train_set[i:i+BATCH])
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(pixel_values=pixels, decoder_input_ids=dec_in).logits
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)
        opt.zero_grad()
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        losses.append(loss.item())
    acc, cer, _ = evaluate(val_set)
    print(f"에폭 {ep+1}/{EPOCHS} · 손실 {sum(losses[-20:])/len(losses[-20:]):.3f} · "
          f"검증 문자열 정확도 {acc*100:.0f}% · CER {cer*100:.1f}% · {time.time()-t0:.0f}초")

plt.figure(figsize=(7, 3))
plt.plot(losses); plt.xlabel("학습 단계 (배치)"); plt.ylabel("손실"); plt.title("학습 손실")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# 셀 5 — 파인튜닝 후 성능 : 전후 비교
acc_after, cer_after, preds_after = evaluate(val_set)
print(f"{'':12}{'문자열 정확도':>10}{'CER':>10}")
print(f"{'파인튜닝 전':10}{acc_before*100:>12.0f}%{cer_before*100:>9.1f}%")
print(f"{'파인튜닝 후':10}{acc_after*100:>12.0f}%{cer_after*100:>9.1f}%")

print("\n같은 이미지, 전과 후")
for (t, _), p0, p1 in list(zip(val_set, preds_before, preds_after))[:10]:
    print(f"  정답 {t}   전 {p0:<14} {'✔' if p0 == t else '✘'}   후 {p1:<14} {'✔' if p1 == t else '✘'}")
show_examples(val_set, preds_after, "파인튜닝 후 — 검증 이미지")

---

## 셀 6 — 새 이미지 읽어 보기

- `TEXTS` 에 아무 일련번호나 넣으면 새로 그려서 읽습니다.
- 직접 찍은 사진이 있으면 `MY_IMAGE` 에 경로를 넣습니다. 글자 한 줄만 보이도록 **잘라서** 넣어야 합니다.
- 형식이 다른 글자(소문자, 긴 문장)를 넣어 보면, 파인튜닝한 형식에 맞춰져 버린 것도 볼 수 있습니다.

In [ ]:
# 셀 6 — 새 이미지 읽어 보기
TEXTS    = ["DL-2026-AI", "OB-0808-ZO", "QZ-1234-B8"]
MY_IMAGE = ""          # 예: "my_label.jpg"

r = random.Random(7)
samples = [(t, draw_label(t, r)) for t in TEXTS]
if MY_IMAGE and os.path.exists(MY_IMAGE):
    samples.append(("(내 사진)", Image.open(MY_IMAGE).convert("RGB")))
preds = read_images([im for _, im in samples])

fig, axes = plt.subplots(1, len(samples), figsize=(3.4 * len(samples), 1.8))
for ax, (t, im), p in zip(list(axes) if len(samples) > 1 else [axes], samples, preds):
    ax.imshow(im); ax.axis("off"); ax.set_title(f"정답 {t}\n예측 {p}", fontsize=10)
plt.tight_layout(); plt.show()
for (t, _), p in zip(samples, preds):
    print(f"{t:>12} → {p!r}")

In [ ]:
# 셀 7 — (선택) 파인튜닝한 모델 저장
SAVE_DIR = "models/ocr_serial"
model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
print("저장:", SAVE_DIR, "→ 다시 쓸 때 VisionEncoderDecoderModel.from_pretrained(SAVE_DIR)")